# The Unitarity Triangle Treasure Hunt
### A hands-on lesson for "Flavour at LHC" students

---

## 🕵️ Your Mission

Somewhere in the complex plane, hidden at mysterious coordinates $(\bar\rho,\,\bar\eta)$, lies the **apex of the Unitarity Triangle** — the key to understanding *CP violation* in the Standard Model.

Your job: act as a particle-physics **detective**. Nature has scattered clues through experiments at Belle, LHCb, BaBar, NA48, and more. Each measurement draws a **band on the map**; when all bands overlap, **X marks the spot**.

---

## 🗺️ How the Hunt Works

| Step | What you do |
|------|-------------|
| 📖 Read the lore | Each chapter gives you theory background |
| 🔐 Crack the code | Fill in the missing Python functions |
| 🧪 Run the check | Auto-grader tells you if you're right |
| 📍 Watch the map | Your band appears on the treasure map |
| 🏆 Find the apex | When all clues converge — you win! |

---

## 🏅 Scoring

- ✅ Correct exercise: **+10 points**
- ⭐ Bonus challenge: **+5 points**
- 🏆 Full triangle revealed: **+20 points**
- Max score: **100 points**

---

> *" A clever citation would fit nicely here! "* by Anonymous

Run **Cell 2** to set up your equipment and start the hunt!

In [ ]:
# ===========================================================
#  MISSION EQUIPMENT — run this first!
# ===========================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.patches import Arc
from scipy.optimize import minimize
from scipy.stats import chi2 as chi2_dist
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 12,
    'axes.labelsize': 14, 'axes.titlesize': 14,
    'figure.dpi': 110, 'text.usetex': False,
})

# Physical / hadronic constants (PDG 2022 + FLAG 2022 lattice)
C = {
    'GF'       : 1.1663788e-5,   # Fermi constant [GeV^-2]
    'mW'       : 80.377,          # W mass [GeV]
    'mt'       : 163.0,           # top MS-bar mass [GeV]
    'mc'       : 1.27,            # charm MS-bar mass [GeV]
    'mK0'      : 0.497611,        # K^0 mass [GeV]
    'DmK'      : 3.484e-15,       # Delta m_K [GeV]
    'FK'       : 0.1561,          # Kaon decay constant [GeV]
    'BKhat'    : 0.7625,          # B_K hat (FLAG 2022)
    'eta_cc'   : 1.87,
    'eta_tt'   : 0.5765,
    'eta_ct'   : 0.496,
    'kappa_eps': 0.940,
    # B mixing
    'mBd'      : 5.27965,
    'FBd_sqBd' : 0.2274,
    'mBs'      : 5.36688,
    'FBs_sqBs' : 0.2742,
    'xi'       : 1.206,
    'etaB'     : 0.5510,
}


# ── Score tracker ────────────────────────────────────────────
class TreasureHunt:
    def __init__(self):
        self.score = 0
        self.clues = []
        self._bands = {}   # name -> (F, val, err, colour)

    def award(self, points, name):
        self.score += points
        self.clues.append(name)
        print(f"  ✅  Clue decoded: '{name}'  +{points} pts")
        print(f"  🏅  Running total: {self.score} pts\n")

    def status(self):
        bar = "█" * self.score + "░" * max(0, 100 - self.score)
        print(f"\n{'='*52}")
        print(f"  🗺️   OPERATION CKM — STATUS REPORT")
        print(f"{'='*52}")
        print(f"  Score  [{bar[:30]}]  {self.score}/100")
        print(f"  Task solved : {len(self.clues)}/7")
        for c in self.clues:
            print(f"    ✓ {c}")
        remaining = 6 - len(self.clues)
        if remaining:
            print(f"  ⏳  {remaining} clue(s) still hidden…")
        else:
            print("  🏆  ALL CLUES FOUND — time to dig!")
        print(f"{'='*52}\n")

hunt = TreasureHunt()

# ── Live treasure map ────────────────────────────────────────
RHO_LIM = (-0.20, 1.1)
ETA_LIM = (-0.10, 0.75)

_rho_v = np.linspace(*RHO_LIM, 300)
_eta_v = np.linspace(*ETA_LIM, 300)
RHO_G, ETA_G = np.meshgrid(_rho_v, _eta_v)

def show_map(title="🗺️  Treasure Map", new=True):
    """Re-draw the accumulating treasure map."""
    fig, ax = plt.subplots(figsize=(8, 6.5))
    ax.set_xlim(*RHO_LIM); ax.set_ylim(*ETA_LIM)
    ax.set_xlabel(r'$\bar{\rho}$', fontsize=15)
    ax.set_ylabel(r'$\bar{\eta}$', fontsize=15)
    ax.set_title(title, fontsize=13)
    ax.grid(True, alpha=0.25)
    ax.axhline(0, color='k', lw=0.4)
    ax.axvline(0, color='k', lw=0.4)
    # watermark
    ax.text(0.15, 0.38, "?", fontsize=80, alpha=0.20, ha='center',
            color='red', fontweight='bold')
    legend_handles = []
    for name, (fn, val, err, col) in hunt._bands.items():
        F = np.vectorize(fn)(RHO_G, ETA_G)
        mask = np.abs(F - val) < err
        ax.contourf(RHO_G, ETA_G, mask.astype(float),
                    levels=[0.5, 1.5], colors=[col], alpha=0.30)
        ax.contour(RHO_G, ETA_G, F, levels=[val],
                   colors=[col], linewidths=1.6, linestyles='--')
        legend_handles.append(
            mpatches.Patch(facecolor=col, alpha=0.5, label=name))
    if legend_handles:
        ax.legend(handles=legend_handles, loc='upper right', fontsize=10)
    plt.tight_layout(); plt.show()

show_map("Treasure Map  (clues will appear here as you solve exercises)")
print("✅  Equipment ready — good luck, detective!")
print(f"   numpy {np.__version__}  |  scipy available")

---
## 📜 Chapter 1 — The Archive: Reading the CKM Matrix

Before the hunt can begin, you need the **decoder key**.

### The CKM Matrix

Quark flavour mixing is described by the **Cabibbo-Kobayashi-Maskawa (CKM) matrix**:

$$
\begin{pmatrix} d' \\ s' \\ b' \end{pmatrix}
=
\begin{pmatrix}
V_{ud} & V_{us} & V_{ub} \\
V_{cd} & V_{cs} & V_{cb} \\
V_{td} & V_{ts} & V_{tb}
\end{pmatrix}
\begin{pmatrix} d \\ s \\ b \end{pmatrix}
$$

Unitarity ($V^\dagger V = \mathbf{1}$) gives nine conditions.  The most useful one — the *first × third column* — is:

$$\boxed{V_{ud}V_{ub}^* + V_{cd}V_{cb}^* + V_{td}V_{tb}^* = 0}$$

Divide everything by $V_{cd}V_{cb}^*$ and plot in the complex plane: you get the famous **Unitarity Triangle**.

Therefore we get the following formula:
$$\boxed{\frac{V_{ud}V_{ub}^*}{V_{cd}V_{cb}^*} + \frac{V_{td}V_{tb}^*}{V_{cd}V_{cb}^*} + 1 = 0}$$

One side is equal to 1 by construction, so the other two sides are:
$$R_b = \left|\frac{V_{ud}V_{ub}^*}{V_{cd}V_{cb}^*}\right|, \quad R_t = \left|\frac{V_{td}V_{tb}^*}{V_{cd}V_{cb}^*}\right|$$

and the angles are:
The three angles:
$$
\beta = \arg\!\left(-\frac{V_{cd}V_{cb}^*}{V_{td}V_{tb}^*}\right), \quad
\gamma = \arg\!\left(-\frac{V_{ud}V_{ub}^*}{V_{cd}V_{cb}^*}\right) = \arctan\!\frac{\bar\eta}{\bar\rho}, \quad
\alpha = \pi - \beta - \gamma
$$



In [ ]:
# draw the triangle identifying the sides and the internal angles

def _angle_arc(ax, vtx, p1, p2, r=0.09, col='black', lbl=''):
    v1 = (p1-vtx)/np.linalg.norm(p1-vtx)
    v2 = (p2-vtx)/np.linalg.norm(p2-vtx)
    t1 = np.degrees(np.arctan2(v1[1],v1[0]))
    t2 = np.degrees(np.arctan2(v2[1],v2[0]))
    if t2 < t1: t1, t2 = t2, t1
    arc = Arc(vtx, 2*r, 2*r, angle=0, theta1=t1, theta2=t2, color=col, lw=1.8)
    ax.add_patch(arc)
    tm = np.radians((t1+t2)/2)
    ax.text(vtx[0]+1.7*r*np.cos(tm), vtx[1]+1.7*r*np.sin(tm),
            lbl, ha='center', va='center', fontsize=16,
            color=col, fontweight='bold')

fig, ax = plt.subplots(figsize=(8, 6.5))
ax.set_xlim(*RHO_LIM); ax.set_ylim(*ETA_LIM)
ax.set_xlabel(r'$\bar{\rho}$', fontsize=15)
ax.set_ylabel(r'$\bar{\eta}$', fontsize=15)
ax.grid(True, alpha=0.25)
ax.axhline(0, color='k', lw=0.4)
ax.axvline(0, color='k', lw=0.4)

temp_rb, temp_eb = 0.166, 0.384
O = np.array([0.0, 0.0])
A_v = np.array([1.0, 0.0])
B_v = np.array([temp_rb, temp_eb])

triangle = plt.Polygon([O, A_v, B_v], fill=True, facecolor='gold', alpha=0.25, edgecolor='navy', linewidth=2.5)
_angle_arc(ax, A_v, O, B_v, r=0.09, col='royalblue', lbl=f'β')
_angle_arc(ax, O,   A_v, B_v, r=0.09, col='purple', lbl=f'γ')
_angle_arc(ax, B_v, O, A_v, r=0.09, col='darkorange', lbl=f'α')


mid_OB = (O+B_v)/2
mid_AB = (A_v+B_v)/2
ax.text(mid_OB[0]-0.045, mid_OB[1]+0.02,
        r'$R_b=\frac{|V_{ud}||V_{ub}^*|}{|V_{cd}||V_{cb}^*|}$', ha='right', fontsize=18, color='forestgreen')
ax.text(mid_AB[0]+0.07, mid_AB[1]+0.02,
        r'$R_t = \frac{|V_{td}||V_{tb}^*|}{|V_{cd}||V_{cb}^*|}$', ha='left', fontsize=18, color='crimson')


ax.text(temp_rb, temp_eb+0.02, r'$(\bar\rho,\bar\eta)$', ha='center', fontsize=18)
ax.add_patch(triangle)
plt.tight_layout()
plt.show()


### The Wolfenstein Parametrisation

Four real parameters $(\lambda, A, \bar\rho, \bar\eta)$ encode the whole matrix:

$$
V_\text{CKM} \approx
\begin{pmatrix}
1 - \tfrac{\lambda^2}{2} & \lambda & A\lambda^3(\rho - i\eta) \\
-\lambda & 1 - \tfrac{\lambda^2}{2} & A\lambda^2 \\
A\lambda^3(1 - \rho - i\eta) & -A\lambda^2 & 1
\end{pmatrix}
+ \mathcal{O}(\lambda^4)
$$

The **rescaled** apex coordinates $(\bar\rho, \bar\eta)$ with $\bar\rho = \rho(1-\lambda^2/2)$, $\bar\eta = \eta(1-\lambda^2/2)$ are our **hidden treasure**.

Key magnitudes you will use throughout the hunt:

| Element | Expression |
|---------|-----------|
| $\|V_{us}\|$ | $\lambda$ |
| $\|V_{cb}\|$ | $A\lambda^2$ |
| $\|V_{ub}\|$ | $A\lambda^3 \sqrt{\bar\rho^2+\bar\eta^2}$ |
| $\|V_{td}\|$ | $A\lambda^3 \sqrt{(1-\bar\rho)^2+\bar\eta^2}$ |

The sides of the triangle can be then expressed as:
$$R_b = \sqrt{\bar\rho^2 + \bar\eta^2}, \quad R_t = \sqrt{(1-\bar\rho)^2 + \bar\eta^2}$$




## 🔐 Clue 1 — The Beauty Scroll: $|V_{ub}/V_{cb}|$ and the ring of power

**Transmission received from HFLAV troll:**

> *"Inclusive and exclusive semileptonic B decays reveal:*
> $\left|\frac{V_{ub}}{V_{cb}}\right|_\text{exp} = xxx \pm xxx $ *"*

### What does this constrain?

$$\left|\frac{V_{ub}}{V_{cb}}\right| = \lambda \sqrt{\bar\rho^2 + \bar\eta^2} = \lambda\, R_b$$

So the measurement $|V_{ub}/V_{cb}| = r \pm \sigma$ draws an **annulus** (ring) in the $(\bar\rho,\bar\eta)$ plane centred at the **origin** with radius $R_b = r/\lambda$.

**Exercise 2**: implement the theoretical prediction and add this clue to the map.

In [ ]:
# ==========================================================
#  EXERCISE 1  — Rb arc (|Vub/Vcb| constraint)
# ==========================================================

# Experimental value
VubVcb_exp = 0
VubVcb_err = 0
lam_ref    = 0.2250     # we use the central value here

def VubVcb_theory(rhobar, etabar, lam=lam_ref):
    """
    Return the SM prediction for |Vub/Vcb| as a function of (rhobar, etabar).
    """
    ### YOUR FORMULA HERE ###
    raise NotImplementedError("Fill in |Vub/Vcb| prediction")

In [ ]:
# ==========================================================
#  CHECK — Exercise 1  +  reveal first band on map
# ==========================================================
try:
    _got  = VubVcb_theory(0.150, 0.357)
    _exp  = lam_ref * np.sqrt(0.150**2 + 0.357**2)
    if abs(_got - _exp) / _exp < 0.005:
        print(f"  ✅  VubVcb_theory(0.15, 0.357) = {_got:.5f}  ✓")

        # Register band on map
        hunt._bands[r"$|V_{ub}/V_{cb}|$"] = (
            lambda r, e: VubVcb_theory(r, e),
            VubVcb_exp, VubVcb_err, "forestgreen"
        )
        hunt.award(15, "|Vub/Vcb| circle")
        show_map("🗺️  Map after Clue 2 — The Rb Circle")
    else:
        print(f"  ❌  Got {_got:.5f}, expected {_exp:.5f} — check your formula")
except NotImplementedError as e:
    print(f"  ⚠️  Not implemented: {e}")

## 🔐 Clue 2 — The Golden Meson: $\sin 2\beta$ from $B \to J/\psi\, K^0$

**Transmission received from Princess Belle, Prince BaBar and duke LHCb:**

> *"Time-dependent CP asymmetry in $B^0 \to J/\psi K^0$:*
> $$\sin 2\beta = xxx \pm xxx$$
> *This is the cleanest CP-violation measurement — essentially no hadronic uncertainty!"*

### What does this constrain?

The angle $\beta$ of the Unitarity Triangle satisfies:

$$\sin 2\beta = \frac{2\,\bar\eta\,(1-\bar\rho)}{(1-\bar\rho)^2 + \bar\eta^2}$$

This is a **hyperbola-like curve** in the $(\bar\rho,\bar\eta)$ plane.  
Notice: it is *independent of $\lambda$ and $A$* — purely geometric!

**Exercise 3**: implement the theoretical prediction.


In [ ]:
# ==========================================================
#  EXERCISE 2  — sin(2β) constraint
# ==========================================================

sin2b_exp = 0
sin2b_err = 0

def sin2beta_theory(rhobar, etabar):
    """
    SM prediction for sin(2β).
    """
    ### YOUR FORMULA HERE ###
    raise NotImplementedError("Fill in sin(2β)")


In [ ]:
# ==========================================================
#  CHECK — Exercise 2  +  reveal band on map
# ==========================================================
try:
    _got = sin2beta_theory(0.150, 0.357)
    _exp = 2*0.357*(1-0.150) / ((1-0.150)**2 + 0.357**2)
    if abs(_got - _exp) / _exp < 0.005:
        print(f"  ✅  sin2beta_theory(0.15, 0.357) = {_got:.4f}  ✓")
        hunt._bands[r"$\sin 2\beta$"] = (
            sin2beta_theory, sin2b_exp, sin2b_err, "royalblue"
        )
        hunt.award(15, "sin(2β) golden meson")
        show_map("🗺️  Map after Clue 3 — The Golden Meson")
    else:
        print(f"  ❌  Got {_got:.4f}, expected {_exp:.4f}")
except NotImplementedError as e:
    print(f"  ⚠️  Not implemented: {e}")

## 🔐 Clue 3 — The Kaon Oracle: $\varepsilon_K$

**Transmission received from NA48 / KTeV:**

> *"Indirect CP violation in the neutral kaon system:*
> $$|\varepsilon_K| = (xxx \pm xxx) \times 10^{-3}$$
> *This tiny number encodes the very same imaginary part of the CKM matrix!"*

### The theoretical expression

After integrating out the heavy quarks (box diagrams), the leading term is:

$$|\varepsilon_K| \approx C_\varepsilon\, \hat{B}_K\, A^2\lambda^6\,\bar\eta
\left[\eta_{tt}\, S_0(x_t)\, A^2\lambda^4(1-\bar\rho)
    + \eta_{ct}\, S_0(x_c,x_t)
    - \eta_{cc}\, S_0(x_c)\right] \kappa_\varepsilon$$

where $S_0$ are **Inami-Lim loop functions** and $\hat{B}_K$ is a lattice QCD input.

Most of the complexity lives in hadronic constants — the key CKM dependence is through $\bar\eta\,[A^2\lambda^4(1-\bar\rho)]$.  
The dominant shape in the $(\bar\rho,\bar\eta)$ plane looks like a *hyperbolic arc*.

**Exercise 4**: complete the `epsilon_K_theory` function using the pre-defined constants below.

In [ ]:
# ==========================================================
#  EXERCISE 3  — ε_K theoretical prediction
# ==========================================================

def _S0t(xt):
    """Inami-Lim top function."""
    return ((4*xt - 11*xt**2 + xt**3) / (4*(1-xt)**2)
            - 3*xt**3*np.log(xt) / (2*(1-xt)**3))

def _S0ct(xc, xt):
    """Inami-Lim top-charm mixed function (approximate)."""
    return xc * (np.log(xt/xc) - 3*xt/(4*(1-xt))
                  - 3*xt**2*np.log(xt)/(4*(1-xt)**2))

epsK_exp = 0
epsK_err = 0   # stat + theory combined

def epsilon_K_theory(rhobar, etabar, lam=0.2250, A=0.826):
    """
    SM prediction for |epsilon_K|.

    The formula is (all inputs from the dict C above):

        xt   = (C['mt'] / C['mW'])**2
        xc   = (C['mc'] / C['mW'])**2
        CeK  = C['GF']**2 * C['FK']**2 * C['mK0'] * C['mW']**2
                 / (6 * sqrt(2) * pi**2 * C['DmK'])

        eps = CeK * C['BKhat'] * A**2 * lam**6 * etabar
              * (C['eta_tt'] * S0t * A**2 * lam**4 * (1 - rhobar)
                 + C['eta_ct'] * S0ct
                 - C['eta_cc'] * xc)
              * C['kappa_eps']

        return abs(eps)
    """
    ### YOUR CODE HERE ###
    # Hint: compute xt, xc, S0t, S0ct, CeK, then assemble eps
    raise NotImplementedError("Fill in epsilon_K_theory")

In [ ]:
# ==========================================================
#  CHECK — Exercise 3  +  reveal ε_K band on map
# ==========================================================
try:
    _got = epsilon_K_theory(0.150, 0.357)
    _exp_range = (1.8e-3, 2.8e-3)   # loose sanity range
    if _exp_range[0] < _got < _exp_range[1]:
        print(f"  ✅  ε_K theory = {_got:.4e}  (exp: 2.228e-3)  ✓")
        # Wrap with fixed lam, A for the map function
        def _epsK_map(rhobar, etabar):
            return epsilon_K_theory(rhobar, etabar)
        hunt._bands[r"$\varepsilon_K$"] = (
            _epsK_map, epsK_exp, epsK_err, "darkorange"
        )
        hunt.award(15, "ε_K Kaon Oracle")
        show_map("🗺️  Map after Clue 4 — The Kaon Oracle")
    else:
        print(f"  ❌  Got {_got:.4e} — outside expected range {_exp_range}")
        print("      Check the CeK prefactor and the S0 functions.")
except NotImplementedError as e:
    print(f"  ⚠️  Not implemented: {e}")

## 🔐 Clue 4 — The Mixing Trail: $\Delta m_d / \Delta m_s$

**Transmission received from HFLAV lord:**

> *"Neutral B meson oscillation frequencies:*
> $$\Delta m_d = xxx \pm xxx~\text{ps}^{-1}$$
> $$\Delta m_s = xxx \pm xxx~\text{ps}^{-1}$$
> *Hadronic uncertainties cancel beautifully in the ratio!"*

### The theoretical ratio

$$\frac{\Delta m_d}{\Delta m_s} = \frac{m_{B_d}}{m_{B_s}} \cdot \frac{1}{\xi^2} \cdot \lambda^2 \left[(1-\bar\rho)^2 + \bar\eta^2\right]$$

where $\xi = f_{B_s}\sqrt{\hat{B}_{B_s}} / (f_{B_d}\sqrt{\hat{B}_{B_d}}) = 1.206 \pm 0.017$ from lattice QCD.

This constrains the **side $R_t$** of the triangle — another arc, centred at $(1, 0)$.

**Exercise 5**: implement the ratio.


In [ ]:
# ==========================================================
#  EXERCISE 4  — Δmd / Δms ratio
# ==========================================================

Dmd_exp = 0;  Dmd_err = 0
Dms_exp = 0;  Dms_err = 0

DmRatio_exp = Dmd_exp / Dms_exp
DmRatio_err = DmRatio_exp * np.sqrt((Dmd_err/Dmd_exp)**2 + (Dms_err/Dms_exp)**2)

def DmRatio_theory(rhobar, etabar, lam=0.2250):
    """
    SM prediction for Δmd / Δms.

    All masses and xi are in the dict C defined in the previous exercise cell.
    """
    ### YOUR FORMULA HERE ###
    raise NotImplementedError("Fill in DmRatio_theory")

In [ ]:
# ==========================================================
#  CHECK — Exercise 4  +  reveal Rt arc on map
# ==========================================================
try:
    _got = DmRatio_theory(0.150, 0.357)
    _exp = (C['mBd']/C['mBs']) * (0.2250**2 * ((1-0.150)**2 + 0.357**2)) / C['xi']**2
    if abs(_got - _exp) / _exp < 0.01:
        print(f"  ✅  Δmd/Δms theory = {_got:.5f}  (exp: {DmRatio_exp:.5f})  ✓")
        hunt._bands[r"$\Delta m_d/\Delta m_s$"] = (
            lambda r, e: DmRatio_theory(r, e),
            DmRatio_exp, DmRatio_err, "crimson"
        )
        hunt.award(15, "Δmd/Δms mixing trail")
        show_map("🗺️  Map after Clue 5 — The Mixing Trail")
    else:
        print(f"  ❌  Got {_got:.5f}, expected {_exp:.5f}")
except NotImplementedError as e:
    print(f"  ⚠️  Not implemented: {e}")


## 🔐 Clue 5 — The Final Angle: $\gamma$ from $B \to DK$

**Transmission received from duke LHCb:**

> *"Direct CP asymmetry in $B^- \to D^0 K^-$ (ADS + GGSZ + GLW combination):*
> $$\gamma = (xxx \pm xxx)°$$
> *This is the only CKM angle measured with negligible theoretical uncertainty!"*

### What does this constrain?

$$\gamma = \arctan\!\left(\frac{\bar\eta}{\bar\rho}\right)$$

In the $(\bar\rho,\bar\eta)$ plane this is a **ray from the origin** — it pins down the *direction* to the apex.

**Exercise 6**: implement the $\gamma$ prediction and unlock the final map band.

In [ ]:
# ==========================================================
#  EXERCISE 5  — γ angle constraint
# ==========================================================

gamma_exp = 0   # degrees
gamma_err = 0    # degrees

def gamma_theory_deg(rhobar, etabar):
    """
    SM prediction for γ in degrees.

    Use np.degrees(np.arctan2(...)) to convert radians to degrees.
    """
    ### YOUR FORMULA HERE ###
    raise NotImplementedError("Fill in gamma_theory_deg")

In [ ]:
# ==========================================================
#  CHECK — Exercise 5  +  reveal γ band on map
# ==========================================================
try:
    _got = gamma_theory_deg(0.150, 0.357)
    _exp = np.degrees(np.arctan2(0.357, 0.150))
    if abs(_got - _exp) < 0.1:
        print(f"  ✅  γ_theory(0.15, 0.357) = {_got:.2f}°  (expected {_exp:.2f}°)  ✓")
        hunt._bands[r"$\gamma$"] = (
            gamma_theory_deg, gamma_exp, gamma_err, "purple"
        )
        hunt.award(15, "γ angle from B→DK")
        print()
        hunt.status()
        show_map("🗺️  Map after Clue 6 — All bands revealed!")
    else:
        print(f"  ❌  Got {_got:.2f}°, expected {_exp:.2f}°")
except NotImplementedError as e:
    print(f"  ⚠️  Not implemented: {e}")

## 💥 The Convergence — $\chi^2$ Fit

All six clues are in. Time to combine them into a single **global fit**.

### The strategy

We minimise:

$$\chi^2(\lambda, A, \bar\rho, \bar\eta) = \sum_{i=1}^{N} \frac{\left[\mathcal{O}_i^\text{exp} - \mathcal{O}_i^\text{th}(\lambda,A,\bar\rho,\bar\eta)\right]^2}{\sigma_i^2}$$

**Confidence regions** in the $(\bar\rho,\bar\eta)$ plane (profiling over $\lambda$ and $A$):

| Confidence level | $\Delta\chi^2$ threshold (2 d.o.f.) |
|---|---|
| 68% | 2.30 |
| 95% | 5.99 |
| 99% | 9.21 |

The remaining cells are **pre-written** — just run them in order. Watch the apex emerge!

In [ ]:
# ==========================================================
#  Full theoretical model -- assembled from your exercises
#  Fallback reference implementations are used automatically
#  for any exercise not yet completed, so you can always run
#  the convergence cells regardless of progress.
# ==========================================================

# ── Reference (fallback) implementations ─────────────────────
def _ref_VubVcb(rb, eb, lam=0.2250):
    return lam * np.sqrt(rb**2 + eb**2)

def _ref_sin2b(rb, eb):
    return 2 * eb * (1 - rb) / ((1 - rb)**2 + eb**2)

def _ref_gamma_deg(rb, eb):
    return np.degrees(np.arctan2(eb, rb))

def _ref_epsK(rb, eb, lam=0.2250, A=0.826):
    xt   = (C['mt'] / C['mW'])**2
    xc   = (C['mc'] / C['mW'])**2
    S0t  = _S0t(xt)
    S0ct = _S0ct(xc, xt)
    CeK  = (C['GF']**2 * C['FK']**2 * C['mK0'] * C['mW']**2
            / (6 * np.sqrt(2) * np.pi**2 * C['DmK']))
    eps  = CeK * C['BKhat'] * A**2 * lam**6 * eb * (
        C['eta_tt'] * S0t * A**2 * lam**4 * (1 - rb)
        + C['eta_ct'] * S0ct
        - C['eta_cc'] * xc
    ) * C['kappa_eps']
    return abs(eps)


def _safe(fn, fallback, *args, **kwargs):
    # Call fn(*args); fall back to fallback(*args) if NotImplementedError.
    try:
        return fn(*args, **kwargs)
    except NotImplementedError:
        return fallback(*args, **kwargs)


# ── Additional theory functions (always provided) ─────────────
def DeltaMd_theory(lam, A, rhobar, etabar):
    xt   = (C['mt'] / C['mW'])**2
    S0t  = _S0t(xt)
    hbar = 6.582119569e-13  # GeV*ps
    Vtd2 = (A * lam**3)**2 * ((1 - rhobar)**2 + etabar**2)
    Dmd  = ((C['GF']**2 / (6 * np.pi**2)) * C['etaB'] * C['mBd']
            * C['FBd_sqBd']**2 * C['mW']**2 * S0t * Vtd2)
    return Dmd / hbar


def DeltaMs_theory(lam, A, rhobar, etabar):
    xt   = (C['mt'] / C['mW'])**2
    S0t  = _S0t(xt)
    hbar = 6.582119569e-13
    Vts2 = (A * lam**2)**2 * (1 - lam**2)**2
    Dms  = ((C['GF']**2 / (6 * np.pi**2)) * C['etaB'] * C['mBs']
            * C['FBs_sqBs']**2 * C['mW']**2 * S0t * Vts2)
    return Dms / hbar


def alpha_theory_deg(rhobar, etabar):
    sin2b  = _safe(sin2beta_theory, _ref_sin2b, rhobar, etabar)
    beta_r = np.arcsin(np.clip(sin2b, -1, 1)) / 2
    gam_r  = np.arctan2(etabar, rhobar)
    return np.degrees(np.pi - beta_r - gam_r)


# ── Experimental constraints dict ────────────────────────────
EXP = {
    'lam'    : (0.2250,   0.00067),
    'Vcb'    : (39.36e-3, np.sqrt(0.68e-3**2 + 0.29e-3**2)),
    'VubVcb' : (0.0864,   0.0025),
    'epsK'   : (2.228e-3, np.sqrt(0.011e-3**2 + 0.15e-3**2)),
    'Dmd'    : (0.5065,   np.sqrt(0.0019**2 + 0.017**2)),
    'Dms'    : (17.765,   np.sqrt(0.006**2  + 0.60**2)),
    'sin2b'  : (0.699,    0.017),
    'gamma'  : (66.2,     3.4),
    'alpha'  : (91.9,     3.1),
}


def chi2_total(params):
    lam, A, rb, eb = params
    if lam <= 0 or A <= 0 or eb < 0:
        return 1e10
    th = {
        'lam'    : lam,
        'Vcb'    : A * lam**2,
        'VubVcb' : _safe(VubVcb_theory,    _ref_VubVcb,   rb, eb, lam),
        'epsK'   : _safe(epsilon_K_theory, _ref_epsK,     rb, eb, lam, A),
        'Dmd'    : DeltaMd_theory(lam, A, rb, eb),
        'Dms'    : DeltaMs_theory(lam, A, rb, eb),
        'sin2b'  : _safe(sin2beta_theory,  _ref_sin2b,    rb, eb),
        'gamma'  : _safe(gamma_theory_deg, _ref_gamma_deg, rb, eb),
        'alpha'  : alpha_theory_deg(rb, eb),
    }
    return sum(((EXP[k][0] - th[k]) / EXP[k][1])**2 for k in EXP)


# ── Quick sanity check ───────────────────────────────────────
val = chi2_total([0.225, 0.826, 0.150, 0.357])
print(f"chi^2 at (0.225, 0.826, 0.150, 0.357) = {val:.2f}")
print("(Uses your implementations where available; reference fallbacks otherwise)")

In [ ]:
# ==========================================================
#  Minimisation — find the best-fit apex
# ==========================================================
p0     = [0.225, 0.826, 0.160, 0.350]
bounds = [(0.20, 0.24), (0.78, 0.88), (-0.20, 0.40), (0.20, 0.50)]

result = minimize(chi2_total, p0, method='L-BFGS-B', bounds=bounds,
                  options={'ftol': 1e-14, 'gtol': 1e-10, 'maxiter': 10000})

lam_fit, A_fit, rb_fit, eb_fit = result.x
chi2_min = result.fun
ckm_fit = CKMWolfenstein(lam_fit, A_fit, rb_fit, eb_fit)

print("+==============================================+")
print("|        BEST-FIT CKM PARAMETERS               |")
print("+==============================================+")
print(f"|  λ      = {lam_fit:.5f}                          |")
print(f"|  A      = {A_fit:.4f}                           |")
print(f"|  ρ̄      = {rb_fit:.4f}                           |")
print(f"|  η̄      = {eb_fit:.4f}                           |")
print(f"|  χ²_min = {chi2_min:.3f}  (d.o.f. = {len(EXP)-4})              |")
print(f"|  χ²/dof = {chi2_min/(len(EXP)-4):.3f}                           |")
print("+==============================================+")


### Why profile over $\lambda$ and $A$?

We have four free parameters $(\lambda, A, \bar\rho, \bar\eta)$, but we only care about the **apex position** $(\bar\rho, \bar\eta)$ — the two "parameters of interest".  
The parameters $\lambda$ and $A$ are **nuisance parameters**: necessary to compute the observables, but we do not want their uncertainty to inflate the $(\bar\rho, \bar\eta)$ contours.

The standard frequentist solution is the **profile likelihood**:

$$\chi^2_\text{prof}(\bar\rho, \bar\eta) = \min_{\lambda,\, A}\; \chi^2(\lambda, A, \bar\rho, \bar\eta)$$

At every point of the $(\bar\rho, \bar\eta)$ grid we let $\lambda$ and $A$ slide to their locally optimal values and keep only the minimum $\chi^2$.  
The double loop below does exactly this: for each grid point `(rb, eb)` a fast 2-parameter minimisation over `[lam, A]` is run with tight bounds.

The resulting map $\Delta\chi^2 = \chi^2_\text{prof} - \chi^2_\text{min}$ follows a $\chi^2$ distribution with **2 degrees of freedom** (one for each parameter of interest).  
The confidence-level thresholds therefore come from the inverse CDF:

| Confidence level | $\Delta\chi^2$ threshold | How it is computed |
|---|---|---|
| 68% | 2.30 | `chi2_dist.ppf(0.68, df=2)` |
| 95% | 5.99 | `chi2_dist.ppf(0.95, df=2)` |

These are the same numbers used by the professional CKM fitters (UTfit, CKMfitter).  
The grid is $120 \times 120 = 14\,400$ points, each requiring one minimisation — hence the ~1 min runtime.

In [ ]:
# ==========================================================
#  Δχ² map in (ρ̄, η̄) — profile over λ and A
#  Takes ~ 1 min
# ==========================================================
print("Computing Δχ² grid (profile over λ, A) … this takes ~1 min")

Nrho, Neta = 120, 120
rho_scan = np.linspace(-0.10, 0.40, Nrho)
eta_scan = np.linspace(0.20,  0.55, Neta)
chi2_map = np.zeros((Neta, Nrho))

for i, eb in enumerate(eta_scan):
    for j, rb in enumerate(rho_scan):
        res_ij = minimize(
            lambda la_A: chi2_total([la_A[0], la_A[1], rb, eb]),
            [lam_fit, A_fit],
            method='L-BFGS-B',
            bounds=[(0.21, 0.24), (0.78, 0.89)],
            options={'ftol': 1e-12, 'maxiter': 200}
        )
        chi2_map[i, j] = res_ij.fun

Dchi2 = chi2_map - chi2_min
print(f"Done. Grid χ²_min = {chi2_map.min():.3f}")


## 🏆 X Marks the Spot — Victory!

All clues decoded. All bands drawn. Run the final cell to dig up the treasure and claim your points!

In [ ]:
# ==========================================================
#  🏆  FINAL REVEAL — The Unitarity Triangle
# ==========================================================

dchi2_68 = chi2_dist.ppf(0.68, df=2)
dchi2_95 = chi2_dist.ppf(0.95, df=2)

RHO_F, ETA_F = np.meshgrid(rho_scan, eta_scan)

fig, ax = plt.subplots(figsize=(16, 8))

# ── Fit result in (ρ̄, η̄) plane ───────────────────────

# All bands
colours = ["forestgreen", "royalblue", "darkorange", "crimson", "purple", "teal"]
band_labels = [r"$|V_{ub}/V_{cb}|$", r"$\sin 2\beta$", r"$\varepsilon_K$",
               r"$\Delta m_d/\Delta m_s$", r"$\gamma$"]
band_fns = [
    (lambda r, e: VubVcb_theory(r, e),   VubVcb_exp, VubVcb_err),
    (sin2beta_theory,                     sin2b_exp,  sin2b_err),
    (lambda r, e: epsilon_K_theory(r, e), epsK_exp,   epsK_err),
    (lambda r, e: DmRatio_theory(r, e),   DmRatio_exp,DmRatio_err),
    (gamma_theory_deg,                    gamma_exp,  gamma_err),
]
legend_handles = []
for (fn, val, err), col, lbl in zip(band_fns, colours, band_labels):
    F = np.vectorize(fn)(RHO_F, ETA_F)
    mask = np.abs(F - val) < err
    ax.contourf(RHO_F, ETA_F, mask.astype(float),
                levels=[0.5, 1.5], colors=[col], alpha=0.28)
    ax.contour(RHO_F, ETA_F, F, levels=[val], colors=[col],
               linewidths=1.5, linestyles='--')
    legend_handles.append(mpatches.Patch(facecolor=col, alpha=0.5, label=lbl+' (1σ)'))

# Confidence contours
ax.contourf(RHO_F, ETA_F, Dchi2, levels=[0, dchi2_68],
            colors=['gold'], alpha=0.55)
ax.contour(RHO_F, ETA_F, Dchi2, levels=[dchi2_68],
           colors=['black'], linewidths=2.0)
ax.contour(RHO_F, ETA_F, Dchi2, levels=[dchi2_95],
           colors=['black'], linewidths=1.5, linestyles='dashed')
legend_handles += [
    Line2D([0],[0], color='black', lw=2,   label='68% CL'),
    Line2D([0],[0], color='black', lw=1.5, linestyle='--', label='95% CL'),
    Line2D([0],[0], color='black', marker='*', markersize=11,
           linestyle='None', label='Best fit'),
]

ax.plot(rb_fit, eb_fit, 'k*', markersize=14, zorder=10)
ax.set_xlim(-0.10, 0.40); ax.set_ylim(0.20, 0.55)
ax.set_xlabel(r'$\bar{\rho}$', fontsize=15)
ax.set_ylabel(r'$\bar{\eta}$', fontsize=15)
ax.set_title(r'Unitarity Triangle Fit in $(\bar\rho,\,\bar\eta)$', fontsize=14)
ax.legend(handles=legend_handles, loc='upper right', fontsize=9, framealpha=0.9)
ax.grid(True, alpha=0.25)



plt.suptitle('🏆  Operation CKM — TREASURE FOUND!', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('ckm_treasure_found.pdf', bbox_inches='tight')
plt.show()

# ── Award final points ───────────────────────────────────────
hunt.award(20, "Unitarity Triangle — X marks the spot")
hunt.status()


print(f"  χ²_min = {chi2_min:.2f}   d.o.f. = {len(EXP)-4}"
      f"   p-value = {1-chi2_dist.cdf(chi2_min,len(EXP)-4):.3f}")
print("\n  ✅  All clues consistent — the Standard Model CKM framework holds!")


## Build the CKM toolkit

Your task: build the **CKM toolkit** by filling in the missing formulas below.

### CKMWolfenstein: what each piece does

This class is a compact calculator for the unitarity-triangle observables using the four Wolfenstein inputs:
$\lambda, A, \bar\rho, \bar\eta$.

#### 1) Initialization
- __init__(lam, A, rhobar, etabar): stores the four parameters as class attributes.

#### 2) CKM magnitudes
- Vus(): returns $|V_{us}| = \lambda$.
- Vcb(): should return $|V_{cb}| = A\lambda^2$.
- Vub(): should return $|V_{ub}| = A\lambda^3\sqrt{\bar\rho^2+\bar\eta^2}$.
- Vtd(): should return $|V_{td}| = A\lambda^3\sqrt{(1-\bar\rho)^2+\bar\eta^2}$.

#### 3) Unitarity-triangle angles
- beta(): converts $\sin(2\beta)$ to $\beta$ via $\beta=\tfrac12\arcsin(\sin2\beta)$.
- gamma(): should implement $\gamma = \operatorname{atan2}(\bar\eta,\bar\rho)$.
- alpha(): uses closure of the triangle: $\alpha = \pi - \beta - \gamma$.

#### 4) Unitarity-triangle sides
- Rb(): should return $R_b=\sqrt{\bar\rho^2+\bar\eta^2}$.
- Rt(): should return $R_t=\sqrt{(1-\bar\rho)^2+\bar\eta^2}$.

Use this as your checklist while replacing each `### YOUR FORMULA HERE ###` in the next cell.

In [ ]:
# ==========================================================
#  EXERCISE  — Complete the CKM Wolfenstein toolkit
#  Fill in every line that says   ### YOUR FORMULA HERE ###
# ==========================================================

class CKMWolfenstein:
    """CKM matrix elements from Wolfenstein parameters (lam, A, rhobar, etabar)."""

    def __init__(self, lam, A, rhobar, etabar):
        self.lam    = lam
        self.A      = A
        self.rhobar = rhobar
        self.etabar = etabar

    # ── CKM magnitudes ────────────────────────────────────────
    def Vus(self):
        return self.lam                         # given — this is the definition of λ

    def Vcb(self):
        ### YOUR FORMULA HERE ###
        # |Vcb| = A * λ²
        raise NotImplementedError("Fill in |Vcb|")

    def Vub(self):
        ### YOUR FORMULA HERE ###
        # |Vub| = A * λ³ * sqrt(ρ̄² + η̄²)
        raise NotImplementedError("Fill in |Vub|")

    def Vtd(self):
        ### YOUR FORMULA HERE ###
        # |Vtd| = A * λ³ * sqrt((1 - ρ̄)² + η̄²)
        raise NotImplementedError("Fill in |Vtd|")

    # ── UT angles ─────────────────────────────────────────────
    def beta(self):
        ### YOUR FORMULA HERE ###
        # β = arcsin(2 η̄ (1 - ρ̄) / [(1 - ρ̄)² + η̄²])
        raise NotImplementedError("Fill in β")

    def gamma(self):
        ### YOUR FORMULA HERE ###
        # γ = arctan(η̄ / ρ̄)   (use np.arctan2 for the correct quadrant)
        raise NotImplementedError("Fill in γ")

    def alpha(self):
        return np.pi - self.beta() - self.gamma()

    # ── UT sides ──────────────────────────────────────────────
    def Rb(self):
        ### YOUR FORMULA HERE ###
        # Rb = sqrt(ρ̄² + η̄²)
        raise NotImplementedError("Fill in Rb")

    def Rt(self):
        ### YOUR FORMULA HERE ###
        # Rt = sqrt((1 - ρ̄)² + η̄²)
        raise NotImplementedError("Fill in Rt")

In [ ]:
# ==========================================================
#  Check your implementation
# ==========================================================
_REF = {"Vcb": 0.04163, "Vub": 0.003665, "Vtd": 0.008717,
        "beta": 22.10, "gamma": 67.14, "Rb": 0.3872, "Rt": 0.8891}
_tol = 5e-2   # 5 % relative tolerance

try:
    _ckm = CKMWolfenstein(lam=0.2250, A=0.826, rhobar=0.150, etabar=0.357)
    _checks = {
        "Vcb"   : (_ckm.Vcb(),                    _REF["Vcb"]),
        "Vub"   : (_ckm.Vub(),                    _REF["Vub"]),
        "Vtd"   : (_ckm.Vtd(),                    _REF["Vtd"]),
        "β"     : (np.degrees(_ckm.beta()),       _REF["beta"]),
        "γ (°)" : (np.degrees(_ckm.gamma()),      _REF["gamma"]),
        "Rb"    : (_ckm.Rb(),                     _REF["Rb"]),
        "Rt"    : (_ckm.Rt(),                     _REF["Rt"]),
    }
    passed = 0
    for name, (got, exp) in _checks.items():
        ok = abs(got - exp) / abs(exp) < _tol
        status = "✅" if ok else "❌"
        print(f"  {status}  {name:<8}  got {got:.5f}  expected {exp:.5f}")
        passed += ok

    if passed == len(_checks):
        #hunt.award(10, "Wolfenstein Decoder")
        print("\n🔓 The decoder is ready.\n")
        print("   α + β + γ = "
              f"{np.degrees(_ckm.alpha()+_ckm.beta()+_ckm.gamma()):.2f}°  (by construction is 180º)")
    else:
        print(f"\n  {passed}/{len(_checks)} checks passed — keep trying! 💪")

except NotImplementedError as e:
    print(f"  ⚠️  Not implemented yet: {e}")

In [ ]:
print("+==============================================+")
print("|        BEST-FIT CKM PARAMETERS               |")
print("+==============================================+")
print(f"|  β       = {np.degrees(ckm_fit.beta()):.2f}°                        |")
print(f"|  γ       = {np.degrees(ckm_fit.gamma()):.2f}°                        |")
print(f"|  α       = {np.degrees(ckm_fit.alpha()):.2f}°                        |")
print(f"|  α+β+γ   = {np.degrees(ckm_fit.alpha()+ckm_fit.beta()+ckm_fit.gamma()):.2f}°                       |")
print(f"|  R_t     = {ckm_fit.Rt:0.2f}                                         |")
print(f"|  R_b     = {ckm_fit.Rt:0.2f}                                         |")
print("+==============================================╝")

## Let's plot the result on the map

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6.5))

rb, eb = rb_fit, eb_fit
O = np.array([0.0, 0.0])
A_v = np.array([1.0, 0.0])
B_v = np.array([rb, eb])

triangle = plt.Polygon([O, A_v, B_v], fill=True, facecolor='gold',
                       alpha=0.25, edgecolor='navy', linewidth=2.5)
ax.add_patch(triangle)

# vertex labels
ax.text(-0.04, -0.06, r'$(0,0)$', ha='center', fontsize=12)
ax.text(1.04,  -0.06, r'$(1,0)$', ha='center', fontsize=12)
ax.text(rb,    eb+0.04, r'$(\bar\rho,\bar\eta)$', ha='center', fontsize=12)

# side labels
def _angle_arc(ax, vtx, p1, p2, r=0.09, col='black', lbl=''):
    v1 = (p1-vtx)/np.linalg.norm(p1-vtx)
    v2 = (p2-vtx)/np.linalg.norm(p2-vtx)
    t1 = np.degrees(np.arctan2(v1[1],v1[0]))
    t2 = np.degrees(np.arctan2(v2[1],v2[0]))
    if t2 < t1: t1, t2 = t2, t1
    arc = Arc(vtx, 2*r, 2*r, angle=0, theta1=t1, theta2=t2, color=col, lw=1.8)
    ax.add_patch(arc)
    tm = np.radians((t1+t2)/2)
    ax.text(vtx[0]+1.7*r*np.cos(tm), vtx[1]+1.7*r*np.sin(tm),
            lbl, ha='center', va='center', fontsize=13,
            color=col, fontweight='bold')

beta_d  = np.degrees(ckm_fit.beta())
gamma_d = np.degrees(ckm_fit.gamma())
alpha_d = np.degrees(ckm_fit.alpha())

_angle_arc(ax, A_v, O, B_v, r=0.09, col='royalblue',
           lbl=f'β\n{beta_d:.1f}°')
_angle_arc(ax, O,   A_v, B_v, r=0.09, col='purple',
           lbl=f'γ\n{gamma_d:.1f}°')
_angle_arc(ax, B_v, O,   A_v, r=0.10, col='darkorange',
           lbl=f'α\n{alpha_d:.1f}°')

mid_OB = (O+B_v)/2
mid_AB = (A_v+B_v)/2
ax.text(mid_OB[0]-0.07, mid_OB[1]+0.02,
        f'$R_b$={ckm_fit.Rb():.3f}', ha='right', fontsize=10, color='forestgreen')
ax.text(mid_AB[0]+0.07, mid_AB[1]+0.02,
        f'$R_t$={ckm_fit.Rt():.3f}', ha='left', fontsize=10, color='crimson')

ax.plot(*B_v, 'k*', markersize=14, zorder=10)
ax.set_xlim(-0.15, 1.20); ax.set_ylim(-0.12, 0.55)
ax.set_aspect('equal')
ax.set_xlabel(r'$\bar\rho$', fontsize=14)
ax.set_ylabel(r'$\bar\eta$', fontsize=14)
ax.set_title('The Unitarity Triangle — best-fit apex', fontsize=14)
ax.axhline(0, color='k', lw=0.4); ax.axvline(0, color='k', lw=0.4)
ax.grid(True, alpha=0.25)

info = (f'Best fit:\n'
        f'$\\bar{{\\rho}}={rb_fit:.3f}$\n'
        f'$\\bar{{\\eta}}={eb_fit:.3f}$\n'
        f'α={alpha_d:.1f}°\n'
        f'β={beta_d:.1f}°\n'
        f'γ={gamma_d:.1f}°\n'
        f'α+β+γ={alpha_d+beta_d+gamma_d:.1f}°')
ax.text(0.76, 0.44, info, fontsize=10,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

plt.suptitle('🏆  Operation CKM — TREASURE FOUND!', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('ckm_treasure_found.pdf', bbox_inches='tight')
plt.show()


# ── Pull table ───────────────────────────────────────────────
print("="*68)
print(f"{'Observable':<14} {'Theory':>13} {'Experiment':>14} {'Pull':>7}")
print("─"*68)
th_vals = {
    'λ'        : lam_fit,
    '|Vcb|×10³': A_fit*lam_fit**2*1e3,
    '|Vub/Vcb|': VubVcb_theory(rb_fit, eb_fit),
    'εK×10³'   : epsilon_K_theory(rb_fit, eb_fit)*1e3,
    'Δmd ps⁻¹' : DeltaMd_theory(lam_fit,A_fit,rb_fit,eb_fit),
    'Δms ps⁻¹' : DeltaMs_theory(lam_fit,A_fit,rb_fit,eb_fit),
    'sin(2β)'  : sin2beta_theory(rb_fit,eb_fit),
    'γ (°)'    : gamma_theory_deg(rb_fit,eb_fit),
    'α (°)'    : alpha_theory_deg(rb_fit,eb_fit),
}
exp_vals = {
    'λ'        : (EXP['lam'][0],    EXP['lam'][1]),
    '|Vcb|×10³': (EXP['Vcb'][0]*1e3, EXP['Vcb'][1]*1e3),
    '|Vub/Vcb|': (EXP['VubVcb'][0], EXP['VubVcb'][1]),
    'εK×10³'   : (EXP['epsK'][0]*1e3, EXP['epsK'][1]*1e3),
    'Δmd ps⁻¹' : (EXP['Dmd'][0],   EXP['Dmd'][1]),
    'Δms ps⁻¹' : (EXP['Dms'][0],   EXP['Dms'][1]),
    'sin(2β)'  : (EXP['sin2b'][0],  EXP['sin2b'][1]),
    'γ (°)'    : (EXP['gamma'][0],  EXP['gamma'][1]),
    'α (°)'    : (EXP['alpha'][0],  EXP['alpha'][1]),
}
for name in th_vals:
    th = th_vals[name]; ev, es = exp_vals[name]
    pull = (ev - th) / es
    flag = "  ⚠️" if abs(pull) > 2 else ""
    print(f"  {name:<12} {th:>13.5g} {ev:>14.5g} {pull:>7.2f}{flag}")
print("="*68)
print(f"  χ²_min = {chi2_min:.2f}   d.o.f. = {len(EXP)-4}"
      f"   p-value = {1-chi2_dist.cdf(chi2_min,len(EXP)-4):.3f}")
print("\n  ✅  All clues consistent — the Standard Model CKM framework holds!")

## ⭐ Bonus Challenge — Bayesian Posterior with MCMC

Want extra points? Run a **Metropolis-Hastings** sampler and compare the
Bayesian posterior credible intervals to the frequentist confidence regions.

**Your task**: tune the step sizes so the acceptance rate sits between 20–40%.  
Verify that the posterior median agrees with the frequentist best-fit point.

*(+5 points for running the cell and reporting the posterior median of $\bar\eta$)*

---

### 🤔 What is this asking — in plain English?

So far we used a **frequentist** approach: we minimised $\chi^2$ to find the single *best-fit* point $(\bar\rho_\text{best}, \bar\eta_\text{best})$ and drew confidence contours around it.

Now we try a **Bayesian** approach. Instead of asking *"what is the best point?"*, we ask:

> *"Given all the measurements, what is the full probability distribution of $(\bar\rho, \bar\eta)$?"*

The answer is called the **posterior distribution** $p(\bar\rho, \bar\eta \mid \text{data})$.

---

### 🎲 What is MCMC?

**Markov Chain Monte Carlo (MCMC)** is a way to *draw random samples* from a distribution that you can evaluate but cannot sample from directly.

Think of it like this:

> Imagine you're a mountain climber in a foggy landscape. You can't see the whole map, but at every step you can measure the altitude (probability) under your feet. MCMC is an algorithm that makes you *wander* around the landscape, spending more time in high-altitude regions — so after many steps, your footprints trace out the shape of the mountain.

The specific algorithm here is **Metropolis-Hastings**:

1. Start at some point $\theta_0 = (\lambda, A, \bar\rho, \bar\eta)$
2. **Propose** a random step: $\theta_\text{prop} = \theta_\text{curr} + \mathcal{N}(0, \sigma)$
3. **Accept or reject**: if the new point is more probable, always accept it; if less probable, accept it with probability = (new prob / old prob)
4. **Record** the accepted point and repeat

After many steps (discarding the first chunk as "burn-in"), the recorded positions form a **chain** that approximates the posterior.

---

### 🔧 Your task

The code below runs the sampler. The key tunable parameter is `step_sizes` — the width of the random jumps in each parameter direction.

- **Too small** → the chain barely moves, gets stuck → acceptance rate near 100%, but explores poorly  
- **Too large** → almost every proposal is rejected → acceptance rate near 0%  
- **Sweet spot** → acceptance rate of **20–40%** is generally good

**Try changing `step_sizes`** in the call to `metropolis_hastings` and re-running until the acceptance rate lands in the 20–40% range.

Then check: does the **posterior median** (the peak of the Bayesian distribution) agree with the **frequentist best-fit** $(\bar\rho_\text{fit}, \bar\eta_\text{fit})$? It should, because we are using flat (uninformative) priors — the two approaches become equivalent in that limit.

In [ ]:
# ==========================================================
#  BONUS — Metropolis-Hastings MCMC
# ==========================================================

def log_post(params):
    """Flat prior x Gaussian likelihood."""
    lam, A, rb, eb = params
    if not (0.20 < lam < 0.24 and 0.78 < A < 0.90
            and -0.20 < rb < 0.40 and 0.10 < eb < 0.55):
        return -np.inf
    return -0.5 * chi2_total(params)

def metropolis_hastings(log_prob, p0, n_steps=80_000,
                        step_sizes=(0.0005, 0.001, 0.005, 0.005),
                        burn_in=10_000, seed=42):
    rng = np.random.default_rng(seed)
    p_curr = np.array(p0, dtype=float)
    lp     = log_prob(p_curr)
    chain  = []
    n_acc  = 0
    for i in range(n_steps):
        p_prop = p_curr + rng.normal(0, step_sizes, size=4)
        lp_new = log_prob(p_prop)
        if np.log(rng.uniform()) < lp_new - lp:
            p_curr, lp = p_prop, lp_new
            n_acc += 1
        if i >= burn_in:
            chain.append(p_curr.copy())
    chain = np.array(chain)
    print(f"  Acceptance rate: {n_acc/n_steps:.1%}  (target: 20–40%)")
    return chain

print("Running MCMC … (~30 s)")
chain = metropolis_hastings(log_post, p0=result.x)

# ── Plot 2D posterior ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

ax = axes[0]
h2d, xe, ye = np.histogram2d(chain[:,2], chain[:,3], bins=60,
                               range=[[-0.05,0.35],[0.25,0.48]])
h2d /= h2d.max()
xc, yc = 0.5*(xe[:-1]+xe[1:]), 0.5*(ye[:-1]+ye[1:])
ax.contourf(xc, yc, h2d.T, levels=[0, 0.32, 0.68, 1.0],
            colors=['white','#90CAF9','#1565C0'], alpha=0.85)
ax.contour(xc, yc, h2d.T, levels=[0.32, 0.68],
           colors=['navy','steelblue'], linewidths=1.5)
ax.plot(rb_fit, eb_fit, 'r*', markersize=12, label='χ² best fit')
ax.set_xlabel(r'$\bar\rho$', fontsize=14); ax.set_ylabel(r'$\bar\eta$', fontsize=14)
ax.set_title('Bayesian posterior', fontsize=13); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.hist(chain[:,3], bins=80, density=True, color='steelblue', alpha=0.7)
lo, hi = np.percentile(chain[:,3], [16, 84])
ax.axvline(lo, color='navy', ls='--', label=f'68% CI: [{lo:.3f}, {hi:.3f}]')
ax.axvline(hi, color='navy', ls='--')
ax.axvline(np.median(chain[:,3]), color='red', label=f'median = {np.median(chain[:,3]):.3f}')
ax.set_xlabel(r'$\bar\eta$', fontsize=14); ax.set_title(r'Marginal posterior $\bar\eta$', fontsize=13)
ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('Bayesian Unitarity Triangle Fit (MCMC)', fontsize=15)
plt.tight_layout(); plt.show()

print(f"\nPosterior medians:  ρ̄ = {np.median(chain[:,2]):.4f},  η̄ = {np.median(chain[:,3]):.4f}")
print(f"Frequentist best:  ρ̄ = {rb_fit:.4f},  η̄ = {eb_fit:.4f}")

hunt.award(5, "Bonus — Bayesian MCMC posterior")
hunt.status()
